# Árbol de Decisión — Regresor

**Repositorio:** `autodidact/both/decision_tree/`

---

## 1. Idea General e Intuición

El **Árbol de Decisión Regresor** funciona igual que el clasificador pero la predicción en cada hoja es la **media del target** de las muestras que caen en esa hoja, en lugar de la clase mayoritaria.

**Propiedad fundamental:** Un árbol de regresión completo (sin restricciones) produce una función **escalonada**: el espacio de features se particiona en regiones rectangulares, y dentro de cada región la predicción es constante.

**Proceso:**
1. Evaluar cada split $(j, s)$ calculando el MSE antes y después de dividir.
2. Elegir el split que más reduce el MSE.
3. Repetir recursivamente hasta condición de parada.
4. Predicción en hoja: $\hat{y} = \bar{y}_{\text{hoja}}$.

## 2. Fundamento Matemático

### 2.1 Criterio de División — Varianza / MSE

Para un nodo $t$ con $n_t$ muestras, el criterio es minimizar el MSE ponderado tras el split:

$$\text{MSE}(t) = \frac{1}{n_t}\sum_{i \in t}(y_i - \bar{y}_t)^2, \quad \bar{y}_t = \frac{1}{n_t}\sum_{i \in t} y_i$$

La ganancia al dividir en $(j, s)$:

$$\Delta\text{MSE}(t, j, s) = \text{MSE}(t) - \frac{n_L}{n_t}\text{MSE}(t_L) - \frac{n_R}{n_t}\text{MSE}(t_R)$$

Se elige el split que **maximiza** $\Delta\text{MSE}$.

### 2.2 Predicción en Hoja

$$\hat{y}(\mathbf{x}) = \bar{y}_{t_{\text{hoja}}(\mathbf{x})} = \frac{1}{|\mathcal{L}|}\sum_{i \in \mathcal{L}} y_i$$

### 2.3 Métricas de Evaluación

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **MSE** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Sensible a outliers (penaliza errores grandes al cuadrado) |
| **RMSE** | $\sqrt{\text{MSE}}$ | Mismas unidades que $y$ |
| **MAE** | $\frac{1}{n}\sum\|y_i - \hat{y}_i\|$ | Robusto a outliers |
| **$R^2$** | $1 - \frac{\text{SS}_\text{res}}{\text{SS}_\text{tot}}$ | 1=perfecto, 0=predice la media, <0=peor que la media |

### 2.4 Cost-Complexity Pruning en Regresión

Igual que en clasificación, pero usando el MSE como medida de error:

$$R_\alpha(T) = \text{MSE}(T) + \alpha \cdot |T|$$

## 3. ¿Cuándo Usarlo y Cuándo NO?

### ✅ Úsalo cuando:
- Las relaciones son **no lineales pero interpretables** (reglas claras).
- Necesitas un modelo **rápido de entrenar** como baseline.
- El target tiene **distribución multimodal** o comportamientos locales.
- Las features tienen **interacciones complejas** entre sí.

### ❌ NO lo uses cuando:
- Necesitas **extrapolación** (un DT solo predice dentro del rango entrenado).
- La relación es **suave y monotónica** (Regresión Lineal/Ridge son mejores).
- El dataset es **ruidoso** (el árbol memorizará el ruido sin poda agresiva).
- Necesitas **alta precisión** (preferir Random Forest o Gradient Boosting).

## 5. Problemas Específicos del Regresor

### 5.1 No Extrapolación (Crítico)
Un árbol de regresión predice la **media de la hoja**. Si el test tiene valores fuera del rango de entrenamiento, el árbol predice la media del extremo más cercano — sin tendencia, sin extrapolar.

### 5.2 Función Escalonada
La predicción es **constante por tramos** (función escalera). Esto es una desventaja si el target varía suavemente.

### 5.3 Outliers en el Target
El criterio MSE es sensible a outliers extremos en el target.  
**Solución:** Usar `criterion='absolute_error'` (MAE) o transformar el target con `log1p`.

### 5.4 Overfitting con Ruido
Un árbol profundo puede hacer una partición solo para ajustar un punto ruidoso.  
**Solución:** `min_samples_leaf ≥ 5` en datasets ruidosos.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
SEED = 42
print('✅ Librerías cargadas.')

---
## 6. Ejemplo 1D — Demostración de la Función Escalonada

In [ ]:
# =============================================================================
# DATOS SINTÉTICOS 1D — Mostrar visualmente la naturaleza escalonada del DT
# Este ejemplo es clave para entender la limitación fundamental del modelo.
# =============================================================================
rng = np.random.RandomState(SEED)
X_1d = np.sort(rng.uniform(0, 10, 200)).reshape(-1, 1)
y_1d = np.sin(X_1d.ravel()) * 3 + rng.randn(200) * 0.4

x_plot = np.linspace(0, 10, 500).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, depth, color in zip(axes, [2, 5, 10], ['steelblue', 'seagreen', 'crimson']):
    dt = DecisionTreeRegressor(max_depth=depth, random_state=SEED)
    dt.fit(X_1d, y_1d)
    y_plot = dt.predict(x_plot)
    ax.scatter(X_1d, y_1d, s=10, alpha=0.5, color='gray', label='Datos')
    ax.plot(x_plot, y_plot, color=color, lw=2, label=f'DT depth={depth}')
    ax.plot(x_plot, np.sin(x_plot.ravel())*3, 'k--', lw=1, alpha=0.5, label='Función real')
    ax.set_title(f'max_depth={depth}\nR²={r2_score(y_1d, dt.predict(X_1d)):.3f} (train)',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8); ax.set_xlabel('x')

plt.suptitle('Función Escalonada del Árbol de Regresión (1D)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# DEMOSTRACIÓN DE NO-EXTRAPOLACIÓN
# El árbol predice la media del nodo extremo, no extrapola la tendencia.
# =============================================================================
X_1d_train = X_1d[X_1d.ravel() <= 8].reshape(-1, 1)
y_1d_train = y_1d[X_1d.ravel() <= 8]

dt_ext = DecisionTreeRegressor(max_depth=5, random_state=SEED)
dt_ext.fit(X_1d_train, y_1d_train)

x_extrap = np.linspace(0, 12, 600).reshape(-1, 1)
y_extrap  = dt_ext.predict(x_extrap)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(X_1d, y_1d, s=10, alpha=0.4, color='gray', label='Datos completos')
ax.axvspan(8, 12, alpha=0.1, color='crimson', label='Zona de extrapolación')
ax.axvline(8, color='crimson', linestyle='--', lw=1.5)
ax.plot(x_extrap, y_extrap, color='steelblue', lw=2.5, label='Predicción DT')
ax.plot(x_extrap, np.sin(x_extrap.ravel())*3, 'k--', lw=1, alpha=0.5, label='Función real')
ax.set_title('No-Extrapolación: El árbol "congela" la predicción en los bordes',
             fontsize=12, fontweight='bold')
ax.legend(); ax.set_xlabel('x')
plt.tight_layout()
plt.show()

---
## 7. Regresión — California Housing

In [ ]:
# =============================================================================
# CARGA Y SPLIT
# =============================================================================
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# =============================================================================
# CURVA BIAS-VARIANZA: max_depth vs MSE (train y test)
# =============================================================================
depths = range(1, 26)
train_mse, test_mse = [], []

for d in depths:
    dt = DecisionTreeRegressor(max_depth=d, random_state=SEED)
    dt.fit(X_train, y_train)
    train_mse.append(mean_squared_error(y_train, dt.predict(X_train)))
    test_mse.append(mean_squared_error(y_test, dt.predict(X_test)))

best_depth = list(depths)[np.argmin(test_mse)]
print(f'Mejor max_depth: {best_depth} | Test RMSE: {np.sqrt(min(test_mse)):.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(depths), np.sqrt(train_mse), 'o-', color='steelblue', lw=2, label='Train RMSE')
ax.plot(list(depths), np.sqrt(test_mse),  's-', color='crimson',   lw=2, label='Test RMSE')
ax.axvline(best_depth, color='gray', linestyle='--', label=f'Mejor depth={best_depth}')
ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title('Curva Bias-Varianza — Árbol de Regresión (California Housing)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# COST-COMPLEXITY PRUNING EN REGRESIÓN
# =============================================================================
path = DecisionTreeRegressor(random_state=SEED).cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]

cv_mse = []
for alpha in ccp_alphas:
    dt_ccp = DecisionTreeRegressor(ccp_alpha=alpha, random_state=SEED)
    sc = cross_val_score(dt_ccp, X_train, y_train, cv=5,
                         scoring='neg_mean_squared_error')
    cv_mse.append(-sc.mean())

best_alpha = ccp_alphas[int(np.argmin(cv_mse))]
print(f'Mejor ccp_alpha: {best_alpha:.6f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(ccp_alphas + 1e-10, np.sqrt(cv_mse), 'o-', color='seagreen', lw=2, ms=4)
ax.axvline(best_alpha + 1e-10, color='crimson', linestyle='--',
           label=f'α={best_alpha:.1e}')
ax.set_xlabel('ccp_alpha (escala log)', fontsize=12)
ax.set_ylabel('CV RMSE (5-fold)', fontsize=12)
ax.set_title('Cost-Complexity Pruning — Regresor', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# MODELO FINAL Y EVALUACIÓN
# =============================================================================
dt_final = DecisionTreeRegressor(max_depth=best_depth, random_state=SEED)
dt_final.fit(X_train, y_train)
y_pred = dt_final.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f'Árbol de Regresión final (max_depth={best_depth}):')
print(f'  Profundidad : {dt_final.get_depth()}')
print(f'  Hojas       : {dt_final.get_n_leaves()}')
print(f'  R²          : {r2:.4f}')
print(f'  RMSE        : {rmse:.4f} (×$100k)')
print(f'  MAE         : {mae:.4f} (×$100k)')

In [ ]:
# =============================================================================
# VISUALIZACIÓN: Predichos vs Reales + Residuos
# =============================================================================
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter predichos vs reales
axes[0].scatter(y_test, y_pred, alpha=0.25, s=8, color='steelblue')
lim = [y_test.min(), y_test.max()]
axes[0].plot(lim, lim, 'r--', lw=2, label='Predicción perfecta')
axes[0].set_xlabel('Valores Reales'); axes[0].set_ylabel('Predicciones')
axes[0].set_title(f'Predichos vs Reales  ($R^2$={r2:.4f})', fontsize=12, fontweight='bold')
axes[0].legend()

# Distribución de residuos
axes[1].hist(residuals, bins=60, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='crimson', lw=2, linestyle='--', label='Error=0')
axes[1].set_xlabel('Residuo (Real - Predicho)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title(f'Distribución de Residuos  (Media={residuals.mean():.4f})',
                  fontsize=12, fontweight='bold')
axes[1].legend()

plt.suptitle('Árbol de Regresión — California Housing',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# IMPORTANCIA DE FEATURES + ÁRBOL (porción)
# =============================================================================
feat_imp = pd.Series(dt_final.feature_importances_, index=housing.feature_names)
feat_imp = feat_imp.sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(feat_imp.index[::-1], feat_imp.values[::-1],
             color='steelblue', alpha=0.8)
axes[0].set_title('Importancia de Features (MDI — MSE)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importancia')

plot_tree(dt_final, max_depth=3, feature_names=housing.feature_names,
          filled=True, rounded=True, impurity=False, ax=axes[1], fontsize=7)
axes[1].set_title(f'Árbol Final (mostrando primeros 3 niveles de {dt_final.get_depth()})',
                  fontsize=12, fontweight='bold')

plt.suptitle('Árbol de Regresión — California Housing', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Comparativa: MSE vs MAE como criterio de división

| Criterio | `criterion='squared_error'` | `criterion='absolute_error'` |
|---|---|---|
| **Minimiza** | MSE en hojas | MAE en hojas |
| **Sensibilidad a outliers** | Alta | Baja |
| **Velocidad** | Más rápido | Más lento |
| **Cuándo usar** | Target sin outliers | Target con outliers extremos |

In [ ]:
# =============================================================================
# COMPARATIVA: squared_error vs absolute_error como criterio
# =============================================================================
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
X_d, y_d = diabetes.data, diabetes.target

X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_d, y_d, test_size=0.2, random_state=SEED
)

for criterion, color in [('squared_error', 'steelblue'), ('absolute_error', 'seagreen')]:
    scores_cv = []
    for d in range(1, 15):
        dt = DecisionTreeRegressor(criterion=criterion, max_depth=d, random_state=SEED)
        sc = cross_val_score(dt, X_tr_d, y_tr_d, cv=5, scoring='neg_mean_squared_error')
        scores_cv.append(np.sqrt(-sc.mean()))
    best_d = list(range(1, 15))[np.argmin(scores_cv)]
    print(f'{criterion:20s} → Mejor depth={best_d}  RMSE={min(scores_cv):.2f}')

print('\nDataset: Diabetes (n=442, 10 features)')

---
## 9. Resumen Comparativo Clasificador vs Regresor

| Aspecto | Clasificador | Regresor |
|---|---|---|
| **Predicción en hoja** | Clase mayoritaria | Media del target |
| **Criterio de split** | Gini / Entropía | MSE / MAE |
| **Métrica principal** | Accuracy / F1 | R² / RMSE |
| **Limitación clave** | Inestabilidad | No extrapola |

### ✅ Checklist Regresor:
- [ ] ¿Grafiqué predichos vs reales y la distribución de residuos?
- [ ] ¿Exploré la curva bias-varianza con `max_depth`?
- [ ] ¿Probé `ccp_alpha` para postpruning?
- [ ] ¿El target requiere extrapolación? (si sí → usar otro modelo)
- [ ] ¿Hay outliers en el target? (si sí → `criterion='absolute_error'`)

---
*Ver notebook de clasificación: `dt_clasificador.ipynb`*